### **Data Loading and Cleaning**

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [2]:
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

In [3]:
db_engine = create_engine(
    f"mysql+pymysql://{user}:{quote_plus(password)}@{host}:{port}",
    echo=True,
    future=True
)

In [4]:
from sqlalchemy import text

with db_engine.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{database}`"))

2026-02-04 12:15:24,679 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-02-04 12:15:24,680 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-04 12:15:24,685 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-02-04 12:15:24,686 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-04 12:15:24,688 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-02-04 12:15:24,689 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-04 12:15:24,692 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-04 12:15:24,694 INFO sqlalchemy.engine.Engine CREATE DATABASE IF NOT EXISTS `student`
2026-02-04 12:15:24,696 INFO sqlalchemy.engine.Engine [generated in 0.00140s] {}
2026-02-04 12:15:24,703 INFO sqlalchemy.engine.Engine COMMIT


In [6]:
db_engine = create_engine(
    f"mysql+pymysql://{user}:{quote_plus(password)}@{host}:{port}/{database}",
    echo=True,
    future=True
)


In [64]:
from sqlalchemy import text

drop_sql = "DROP TABLE IF EXISTS mental_health;"
create_sql = """
CREATE TABLE mental_health (
  id INT NOT NULL,
  gender VARCHAR(10) NULL,
  age TINYINT UNSIGNED NULL,
  city VARCHAR(100) NULL,
  degree VARCHAR(100) NULL,

  academic_pressure TINYINT UNSIGNED NULL,
  cgpa DECIMAL(4,2) NULL,
  study_satisfaction TINYINT UNSIGNED NULL,

  sleep_duration VARCHAR(50) NULL,
  dietary_habits VARCHAR(50) NULL,

  suicidal_thoughts VARCHAR(10) NULL,
  work_study_hours TINYINT UNSIGNED NULL,
  financial_stress TINYINT UNSIGNED NULL,
  family_history_mental_illness VARCHAR(10) NULL,

  depression TINYINT UNSIGNED NULL,
  PRIMARY KEY (id)
);
"""

with db_engine.connect() as conn:
    conn.execute(text(drop_sql))
    conn.execute(text(create_sql))
    conn.commit()


2026-02-03 15:43:01,294 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-02-03 15:43:01,296 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 15:43:01,300 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-02-03 15:43:01,301 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 15:43:01,304 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-02-03 15:43:01,306 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 15:43:01,309 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:01,310 INFO sqlalchemy.engine.Engine DROP TABLE IF EXISTS mental_health;
2026-02-03 15:43:01,311 INFO sqlalchemy.engine.Engine [generated in 0.00222s] {}
2026-02-03 15:43:01,321 INFO sqlalchemy.engine.Engine 
CREATE TABLE mental_health (
  id INT NOT NULL,
  gender VARCHAR(10) NULL,
  age TINYINT UNSIGNED NULL,
  city VARCHAR(100) NULL,
  degree VARCHAR(100) NULL,

  academic_pressure TINYINT UNSIGNED NULL,
  cgpa DECIMAL(4,2) NULL,
  study_satisfaction TINYINT UNSIGNED NULL,


In [65]:
import pandas as pd

df = pd.read_csv("Student Depression Dataset.csv")

text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

df = df.rename(columns={
    "Gender": "gender",
    "Age": "age",
    "City": "city",
    "Academic Pressure": "academic_pressure",
    "CGPA": "cgpa",
    "Study Satisfaction": "study_satisfaction",
    "Sleep Duration": "sleep_duration",
    "Dietary Habits": "dietary_habits",
    "Degree": "degree",
    "Have you ever had suicidal thoughts ?": "suicidal_thoughts",
    "Work/Study Hours": "work_study_hours",
    "Financial Stress": "financial_stress",
    "Family History of Mental Illness": "family_history_mental_illness",
    "Depression": "depression",
    "id": "id"
})

df.columns


Index(['id', 'gender', 'age', 'city', 'academic_pressure', 'cgpa',
       'study_satisfaction', 'sleep_duration', 'dietary_habits', 'degree',
       'suicidal_thoughts', 'work_study_hours', 'financial_stress',
       'family_history_mental_illness', 'depression'],
      dtype='str')

In [66]:
df.to_sql(
    name="mental_health",
    con=db_engine,
    if_exists="append",
    index=False,
    chunksize=1000  
)


2026-02-03 15:43:01,582 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:01,588 INFO sqlalchemy.engine.Engine DESCRIBE `student`.`mental_health`
2026-02-03 15:43:01,589 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 15:43:01,649 INFO sqlalchemy.engine.Engine INSERT INTO mental_health (id, gender, age, city, academic_pressure, cgpa, study_satisfaction, sleep_duration, dietary_habits, degree, suicidal_thoughts, work_study_hours, financial_stress, family_history_mental_illness, depression) VALUES (%(id)s, %(gender)s, %(age)s, %(city)s, %(academic_pressure)s, %(cgpa)s, %(study_satisfaction)s, %(sleep_duration)s, %(dietary_habits)s, %(degree)s, %(suicidal_thoughts)s, %(work_study_hours)s, %(financial_stress)s, %(family_history_mental_illness)s, %(depression)s)
2026-02-03 15:43:01,651 INFO sqlalchemy.engine.Engine [generated in 0.01921s] [{'id': 2, 'gender': 'male', 'age': 33.0, 'city': 'visakhapatnam', 'academic_pressure': 5.0, 'cgpa': 8.97, 'study_satisfaction': 2.0, 

27870

In [67]:
clean_city_sql = [
    # Fix misspellings / noisy values
    "UPDATE mental_health SET city = 'Ghaziabad' WHERE city = 'Khaziabad';",
    "UPDATE mental_health SET city = 'Delhi' WHERE city = 'Less Delhi';",
    "UPDATE mental_health SET city = 'Kalyan' WHERE city = 'Less than 5 Kalyan';",
    "UPDATE mental_health SET city = 'Kalyan' WHERE city = 'Nalyan';",

    # Invalid numeric / header values
    "UPDATE mental_health SET city = 'Unknown' WHERE city IN ('City','3');",

    # Names, degrees, invalid values
    """
    UPDATE mental_health
    SET city = 'Unknown'
    WHERE city IN (
        'Bhavna','Harsh','Harsha','Gaurav','Mihir','Mira','Nalini',
        'Nandini','Rashi','Reyansh','Saanvi','Vaanya',
        'M.Com','M.Tech','ME','Kibara'
    );
    """,

    # Everything else not valid → Unknown
    """
    UPDATE mental_health
    SET city = 'Unknown'
    WHERE city NOT IN (
        'Agra','Ahmedabad','Bangalore','Bhopal','Chennai','Delhi','Faridabad',
        'Ghaziabad','Hyderabad','Indore','Jaipur','Kalyan','Kanpur','Kolkata',
        'Lucknow','Ludhiana','Meerut','Mumbai','Nagpur','Nashik','Patna','Pune',
        'Rajkot','Srinagar','Surat','Thane','Vadodara','Varanasi',
        'Vasai-Virar','Visakhapatnam','Unknown'
    );
    """
]

with db_engine.connect() as conn:
    for query in clean_city_sql:
        conn.execute(text(query))
    conn.commit()

2026-02-03 15:43:04,073 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:04,075 INFO sqlalchemy.engine.Engine UPDATE mental_health SET city = 'Ghaziabad' WHERE city = 'Khaziabad';
2026-02-03 15:43:04,076 INFO sqlalchemy.engine.Engine [generated in 0.00369s] {}
2026-02-03 15:43:04,156 INFO sqlalchemy.engine.Engine UPDATE mental_health SET city = 'Delhi' WHERE city = 'Less Delhi';
2026-02-03 15:43:04,157 INFO sqlalchemy.engine.Engine [generated in 0.00152s] {}
2026-02-03 15:43:04,264 INFO sqlalchemy.engine.Engine UPDATE mental_health SET city = 'Kalyan' WHERE city = 'Less than 5 Kalyan';
2026-02-03 15:43:04,266 INFO sqlalchemy.engine.Engine [generated in 0.00125s] {}
2026-02-03 15:43:04,385 INFO sqlalchemy.engine.Engine UPDATE mental_health SET city = 'Kalyan' WHERE city = 'Nalyan';
2026-02-03 15:43:04,387 INFO sqlalchemy.engine.Engine [generated in 0.00134s] {}
2026-02-03 15:43:04,455 INFO sqlalchemy.engine.Engine UPDATE mental_health SET city = 'Unknown' WHERE city IN ('

* `academic_pressure == 0 → 9 rows (0.032%)`
* `cgpa == 0 → 9 rows (0.032%)`
* `study_satisfaction == 0 → 10 rows (0.036%)`
* `financial_stress is null → 3 rows (0.011%)`

In [68]:
# Count rows that will be deleted
count_sql = """
SELECT COUNT(*)
FROM mental_health
WHERE academic_pressure = 0
   OR cgpa = 0
   OR study_satisfaction = 0
   OR financial_stress IS NULL;
"""

delete_sql = """
DELETE FROM mental_health
WHERE academic_pressure = 0
   OR cgpa = 0
   OR study_satisfaction = 0
   OR financial_stress IS NULL;
"""

with db_engine.begin() as conn:
    rows_to_delete = conn.execute(text(count_sql)).scalar()
    conn.execute(text(delete_sql))

print(f"Rows deleted: {rows_to_delete}")


2026-02-03 15:43:04,714 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:04,716 INFO sqlalchemy.engine.Engine 
SELECT COUNT(*)
FROM mental_health
WHERE academic_pressure = 0
   OR cgpa = 0
   OR study_satisfaction = 0
   OR financial_stress IS NULL;

2026-02-03 15:43:04,717 INFO sqlalchemy.engine.Engine [generated in 0.00137s] {}
2026-02-03 15:43:04,747 INFO sqlalchemy.engine.Engine 
DELETE FROM mental_health
WHERE academic_pressure = 0
   OR cgpa = 0
   OR study_satisfaction = 0
   OR financial_stress IS NULL;

2026-02-03 15:43:04,749 INFO sqlalchemy.engine.Engine [generated in 0.00186s] {}
2026-02-03 15:43:04,830 INFO sqlalchemy.engine.Engine COMMIT
Rows deleted: 17


In [69]:
# Categorize degree values into School, ug, pg, phd, and others

add_col_sql = """
ALTER TABLE mental_health
ADD COLUMN degree_category VARCHAR(20) NULL
AFTER degree;
"""

update_degree_category_sql = """
UPDATE mental_health
SET degree_category = CASE

  -- School
  WHEN degree = 'class 12' THEN 'School'

  -- PhD
  WHEN degree = 'phd' THEN 'phd'

  -- Undergraduate
  WHEN degree IN (
    'ba','bsc','bca','b.com','b.ed','b.arch','b.tech',
    'be','b.pharm','bba','bhm','llb','mbbs'
  ) THEN 'ug'

  -- Postgraduate
  WHEN degree IN (
    'ma','msc','mca','mba','m.com','m.ed','m.tech',
    'me','m.pharm','md','mhm','llm'
  ) THEN 'pg'

  -- Others
  WHEN degree = 'others' THEN 'others'

  ELSE NULL
END;
"""

with db_engine.begin() as conn:
    conn.execute(text(add_col_sql))
    conn.execute(text(update_degree_category_sql))

print("degree_category column created and populated.")


2026-02-03 15:43:04,853 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:04,855 INFO sqlalchemy.engine.Engine 
ALTER TABLE mental_health
ADD COLUMN degree_category VARCHAR(20) NULL
AFTER degree;

2026-02-03 15:43:04,857 INFO sqlalchemy.engine.Engine [generated in 0.00183s] {}
2026-02-03 15:43:04,886 INFO sqlalchemy.engine.Engine 
UPDATE mental_health
SET degree_category = CASE

  -- School
  WHEN degree = 'class 12' THEN 'School'

  -- PhD
  WHEN degree = 'phd' THEN 'phd'

  -- Undergraduate
  WHEN degree IN (
    'ba','bsc','bca','b.com','b.ed','b.arch','b.tech',
    'be','b.pharm','bba','bhm','llb','mbbs'
  ) THEN 'ug'

  -- Postgraduate
  WHEN degree IN (
    'ma','msc','mca','mba','m.com','m.ed','m.tech',
    'me','m.pharm','md','mhm','llm'
  ) THEN 'pg'

  -- Others
  WHEN degree = 'others' THEN 'others'

  ELSE NULL
END;

2026-02-03 15:43:04,888 INFO sqlalchemy.engine.Engine [generated in 0.00171s] {}
2026-02-03 15:43:07,036 INFO sqlalchemy.engine.Engine COMMIT
deg

In [70]:
# Create red_flag column

add_red_flag_sql = """
ALTER TABLE mental_health
ADD COLUMN red_flag TINYINT NULL
AFTER family_history_mental_illness;
"""

update_red_flag_sql = """
UPDATE mental_health
SET red_flag =
    (academic_pressure >= 4) +
    (financial_stress >= 4) +
    (work_study_hours >= 10) +
    (study_satisfaction <= 2) +
    (sleep_duration = 'less than 5 hours') +
    (dietary_habits = 'unhealthy') +
    (suicidal_thoughts = 'yes');
"""

with db_engine.begin() as conn:
    conn.execute(text(add_red_flag_sql))
    conn.execute(text(update_red_flag_sql))

print("red_flag column created and populated.")


2026-02-03 15:43:07,083 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:07,085 INFO sqlalchemy.engine.Engine 
ALTER TABLE mental_health
ADD COLUMN red_flag TINYINT NULL
AFTER family_history_mental_illness;

2026-02-03 15:43:07,086 INFO sqlalchemy.engine.Engine [generated in 0.00123s] {}
2026-02-03 15:43:07,112 INFO sqlalchemy.engine.Engine 
UPDATE mental_health
SET red_flag =
    (academic_pressure >= 4) +
    (financial_stress >= 4) +
    (work_study_hours >= 10) +
    (study_satisfaction <= 2) +
    (sleep_duration = 'less than 5 hours') +
    (dietary_habits = 'unhealthy') +
    (suicidal_thoughts = 'yes');

2026-02-03 15:43:07,114 INFO sqlalchemy.engine.Engine [generated in 0.00127s] {}
2026-02-03 15:43:12,699 INFO sqlalchemy.engine.Engine COMMIT
red_flag column created and populated.


In [ ]:
# Create wellness_drivers column

add_wellness_drivers_sql = """
ALTER TABLE mental_health
ADD COLUMN wellness VARCHAR(20) NULL
AFTER red_flag;
"""

update_wellness_drivers_sql = """
UPDATE mental_health
SET wellness = CASE
  WHEN red_flag <= 2 THEN 'high'
  WHEN red_flag <= 5 THEN 'moderate'
  ELSE 'low'
END;
"""

with db_engine.begin() as conn:
    conn.execute(text(add_wellness_drivers_sql))
    conn.execute(text(update_wellness_drivers_sql))

print("wellness_drivers column created and populated.")


2026-02-03 15:43:12,753 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:12,755 INFO sqlalchemy.engine.Engine 
ALTER TABLE mental_health
ADD COLUMN wellness VARCHAR(20) NULL
AFTER red_flag;

2026-02-03 15:43:12,756 INFO sqlalchemy.engine.Engine [generated in 0.00127s] {}
2026-02-03 15:43:12,789 INFO sqlalchemy.engine.Engine 
UPDATE mental_health
SET wellness = CASE
  WHEN red_flag <= 2 THEN 'high'
  WHEN red_flag <= 4 THEN 'moderate'
  ELSE 'low'
END;

2026-02-03 15:43:12,790 INFO sqlalchemy.engine.Engine [generated in 0.00120s] {}
2026-02-03 15:43:14,403 INFO sqlalchemy.engine.Engine COMMIT
wellness_drivers column created and populated.


In [72]:
# Create support_priority column

add_support_priority_sql = """
ALTER TABLE mental_health
ADD COLUMN support_priority VARCHAR(30) NULL
AFTER wellness;
"""

update_support_priority_sql = """
UPDATE mental_health
SET support_priority = CASE

  -- Depressed students (action first)
  WHEN depression = 1 AND red_flag >= 6
    THEN 'critical'

  WHEN depression = 1 AND red_flag >= 5
    THEN 'high priority'

  WHEN depression = 1
    THEN 'moderate priority'

  -- Not depressed (prevention)
  WHEN depression = 0 AND red_flag >= 5
    THEN 'preventive high risk'

  WHEN depression = 0 AND red_flag >= 3
    THEN 'preventive watchlist'

  ELSE 'stable'
END;
"""

with db_engine.begin() as conn:
    conn.execute(text(add_support_priority_sql))
    conn.execute(text(update_support_priority_sql))

print("support_priority column created and populated.")


2026-02-03 15:43:14,454 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:14,456 INFO sqlalchemy.engine.Engine 
ALTER TABLE mental_health
ADD COLUMN support_priority VARCHAR(30) NULL
AFTER wellness;

2026-02-03 15:43:14,457 INFO sqlalchemy.engine.Engine [generated in 0.00178s] {}
2026-02-03 15:43:14,490 INFO sqlalchemy.engine.Engine 
UPDATE mental_health
SET support_priority = CASE

  -- Depressed students (action first)
  WHEN depression = 1 AND red_flag >= 6
    THEN 'critical'

  WHEN depression = 1 AND red_flag >= 5
    THEN 'high priority'

  WHEN depression = 1
    THEN 'moderate priority'

  -- Not depressed (prevention)
  WHEN depression = 0 AND red_flag >= 5
    THEN 'preventive high risk'

  WHEN depression = 0 AND red_flag >= 3
    THEN 'preventive watchlist'

  ELSE 'stable'
END;

2026-02-03 15:43:14,492 INFO sqlalchemy.engine.Engine [generated in 0.00153s] {}
2026-02-03 15:43:16,149 INFO sqlalchemy.engine.Engine COMMIT
support_priority column created and popu

In [74]:
import pandas as pd

df = pd.read_sql("SELECT * FROM mental_health", db_engine)
df.to_csv("mental_health.csv", index=False)

print("Saved as mental_health.csv")


2026-02-03 15:43:42,226 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-03 15:43:42,229 INFO sqlalchemy.engine.Engine DESCRIBE `student`.`SELECT * FROM mental_health`
2026-02-03 15:43:42,231 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 15:43:42,233 INFO sqlalchemy.engine.Engine SELECT * FROM mental_health
2026-02-03 15:43:42,234 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-02-03 15:43:43,273 INFO sqlalchemy.engine.Engine ROLLBACK
Saved as mental_health.csv
